In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm import tqdm
import numpy as np
import pickle

ImportError: DLL load failed while importing _imaging: Az operációs rendszer nem tudja futtatni a következő programot: %1.

In [38]:
NUM_FOLDS = 5
NUM_CLASSES = 3
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

LMDB_ROOT = "./lmdbs"

In [ ]:
scaler = torch.cuda.amp.GradScaler()

with torch.cuda.amp.autocast():
    outputs = model(images)
    loss = criterion(outputs, labels)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

In [39]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [40]:
import lmdb
import pickle
import io
from PIL import Image
from torch.utils.data import Dataset

class LMDBDataset(Dataset):
    def __init__(self, lmdb_path, transform=None):
        self.lmdb_path = lmdb_path
        self.transform = transform

        # Open once ONLY to read keys, then close
        env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            self.keys = [k for k, _ in txn.cursor() if k != b"__len__"]

        env.close()

        # 🔥 Critical: length derived from keys, not __len__
        self.length = len(self.keys)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError

        # Open LMDB locally (safe for Windows)
        env = lmdb.open(
            self.lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            data = pickle.loads(txn.get(self.keys[idx]))

        env.close()

        img = Image.open(io.BytesIO(data["image"])).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, data["label"]


In [44]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [45]:
fold_results = []

for fold in range(NUM_FOLDS):
    print(f"\n===== Fold {fold} =====")

    train_lmdb = f"{LMDB_ROOT}/fold_{fold}_train.lmdb"
    val_lmdb = f"{LMDB_ROOT}/fold_{fold}_val.lmdb"

    train_dataset = LMDBDataset(train_lmdb, transform=train_transform)
    val_dataset = LMDBDataset(val_lmdb, transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # Model
    model = models.alexnet(pretrained=True)
    model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )

        val_loss, val_acc = validate(
            model, val_loader, criterion
        )

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                model.state_dict(),
                f"alexnet_fold_{fold}.pth"
            )

    fold_results.append(best_val_acc)


===== Fold 0 =====


C:\Users\istva\AppData\Local\Temp\ipykernel_14876\1625626329.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/15


C:\Users\istva\AppData\Local\Temp\ipykernel_14876\2851648072.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\istva\AppData\Local\Temp\ipykernel_14876\2851648072.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Train Loss: 0.1398 | Train Acc: 0.9948 | Val Loss: 19.9881 | Val Acc: 0.0799
Epoch 2/15
Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 28.4141 | Val Acc: 0.0799
Epoch 3/15
Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 31.9498 | Val Acc: 0.0799
Epoch 4/15
Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 33.4102 | Val Acc: 0.0799
Epoch 5/15
Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 33.9849 | Val Acc: 0.0799
Epoch 6/15
Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 34.2121 | Val Acc: 0.0799
Epoch 7/15


KeyboardInterrupt: 

In [46]:
from collections import Counter

def label_distribution(lmdb_path):
    ds = LMDBDataset(lmdb_path)
    labels = [ds[i][1] for i in range(len(ds))]
    return Counter(labels)

print("TRAIN:", label_distribution(train_lmdb))
print("VAL:",   label_distribution(val_lmdb))


TRAIN: Counter({0: 1157})
VAL: Counter({2: 15454, 1: 1657, 0: 1486})


In [1]:
import h5py
import torch
torch.backends.cudnn.benchmark = True
from torch.utils.data import Dataset
import numpy as np
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.nn as nn

class HDF5KFoldDataset(Dataset):
    def __init__(self, h5_path, fold, split, transform=None):
        self.h5_path = h5_path
        self.fold = str(fold)
        self.split = split
        self.transform = transform

        self.data = []
        self.labels = []

        self.class_map = {
            "Normal": 0,
            "COVID-19": 1,
            "Pneumonia": 2
        }

        with h5py.File(self.h5_path, 'r') as f:
            base = f[f"folds/{self.fold}/{self.split}"]

            for cls_name, label in self.class_map.items():
                dataset = base[cls_name]   # <-- this is a Dataset
                num_samples = dataset.shape[0]

                for i in range(num_samples):
                    self.data.append((cls_name, i))
                    self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        cls_name, i = self.data[idx]
        label = self.labels[idx]

        with h5py.File(self.h5_path, 'r') as f:
            img = f[f"folds/{self.fold}/{self.split}/{cls_name}"][i]

        # Convert to float32 and scale
        img = img.astype("float32") / 255.0

        # Handle grayscale vs RGB
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=0)   # (3, H, W)
        elif img.ndim == 3 and img.shape[-1] == 3:
            img = img.transpose(2, 0, 1)        # (3, H, W)

        img = torch.from_numpy(img)

        if self.transform:
            img = self.transform(img)

        return img, label

In [2]:
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [5]:
dataset = HDF5KFoldDataset("alexnet_image_data.h5", 0, "train")

print(len(dataset))

for i in range(5):
    img, label = dataset[i]
    print(i, img.shape, label)


302932
0 torch.Size([3, 227, 227]) 0
1 torch.Size([3, 227, 227]) 0
2 torch.Size([3, 227, 227]) 0
3 torch.Size([3, 227, 227]) 0
4 torch.Size([3, 227, 227]) 0


In [9]:
num_epochs = 10
num_folds = 5
h5_file = 'alexnet_image_data.h5'

for fold in range(num_folds):
    print(f"\n===== Fold {fold} =====")
    model = models.alexnet(weights=True)
    model.classifier[6] = nn.Linear(4096, 3)
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_dataset = HDF5KFoldDataset(h5_file, fold, "train", transform)
    val_dataset   = HDF5KFoldDataset(h5_file, fold, "val", transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_dataset, batch_size=32, num_workers=0, pin_memory=False)

    print("Dataset length:", len(train_dataset))
    img, label = train_dataset[0]
    print("Sample OK")

    for epoch in range(num_epochs):
        loss = train_one_epoch(model, train_loader)
        print("train done")
        acc = validate(model, val_loader)
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, Val Acc={acc:.4f}")


===== Fold 0 =====
Dataset length: 302932
Sample OK


KeyboardInterrupt: 